# Retrieval Drift 감지와 대응

문서가 업데이트되면 동일한 질문에 대한 **검색 결과가 달라진다**.
이것이 **Retrieval Drift**이며, RAG 시스템의 응답 품질을 조용히 떨어뜨린다.

이 실습에서는 문서 변경 전후의 검색 결과를 비교하고, **Stability Score**로 정량 측정한다.

### 학습 목표

1. InMemoryVectorStore로 간단한 RAG 파이프라인을 구축한다
2. 문서 업데이트가 검색 결과에 미치는 영향을 관찰한다
3. Retrieval Stability Score를 계산하고 해석한다
4. Langfuse로 검색 품질 변화를 추적한다

---

In [1]:
from dotenv import load_dotenv
load_dotenv()

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langfuse import Langfuse
from langfuse.langchain import CallbackHandler

## 1. 사내 문서 RAG 파이프라인 구축

HR 정책 문서 6건을 벡터 스토어에 인덱싱한다.

In [2]:
# ── 문서 v1: 현행 정책 ──
docs_v1 = [
    Document(
        page_content="재택근무는 주 2회까지 가능합니다. 팀장 사전 승인이 필요합니다.",
        metadata={"id": "HR-001", "title": "재택근무 정책"},
    ),
    Document(
        page_content="연차는 입사 1년 후 15일이 부여됩니다. 미사용 연차는 다음 해로 이월되지 않습니다.",
        metadata={"id": "HR-002", "title": "연차 정책"},
    ),
    Document(
        page_content="출장비는 교통비, 숙박비, 식비를 포함합니다. 사전 승인 후 법인카드로 결제합니다.",
        metadata={"id": "HR-003", "title": "출장비 정책"},
    ),
    Document(
        page_content="신입사원 교육은 입사 첫 주에 진행됩니다. OJT 2주, 멘토링 3개월이 포함됩니다.",
        metadata={"id": "HR-004", "title": "온보딩 프로그램"},
    ),
    Document(
        page_content="성과평가는 반기 1회 실시합니다. 상사 평가 60%, 동료 평가 20%, 자기 평가 20%입니다.",
        metadata={"id": "HR-005", "title": "성과평가 기준"},
    ),
    Document(
        page_content="건강검진은 연 1회 회사 부담으로 제공됩니다. 배우자 검진비 50%를 지원합니다.",
        metadata={"id": "HR-006", "title": "건강검진 지원"},
    ),
]

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
vectorstore_v1 = InMemoryVectorStore.from_documents(docs_v1, embeddings)
retriever_v1 = vectorstore_v1.as_retriever(search_kwargs={"k": 3})

print(f"✅ 벡터 스토어 v1 구축 완료 — 문서 {len(docs_v1)}건")

✅ 벡터 스토어 v1 구축 완료 — 문서 6건


In [3]:
# ── 기준 검색 결과 저장 (t1) ──
test_queries = [
    "재택근무는 어떻게 신청하나요?",
    "연차 몇 일이에요?",
    "출장비 정산 방법이 궁금합니다",
    "성과평가 비율이 어떻게 되나요?",
]

baseline = {}  # {query: [doc_id, ...]}
for query in test_queries:
    docs = retriever_v1.invoke(query)
    baseline[query] = [d.metadata["id"] for d in docs]
    print(f"Q: {query}")
    for d in docs:
        print(f"   → {d.metadata['id']} | {d.page_content[:50]}...")
    print()

print("📌 기준점(t1) 저장 완료")

Q: 재택근무는 어떻게 신청하나요?
   → HR-001 | 재택근무는 주 2회까지 가능합니다. 팀장 사전 승인이 필요합니다....
   → HR-006 | 건강검진은 연 1회 회사 부담으로 제공됩니다. 배우자 검진비 50%를 지원합니다....
   → HR-003 | 출장비는 교통비, 숙박비, 식비를 포함합니다. 사전 승인 후 법인카드로 결제합니다....

Q: 연차 몇 일이에요?
   → HR-002 | 연차는 입사 1년 후 15일이 부여됩니다. 미사용 연차는 다음 해로 이월되지 않습니다....
   → HR-001 | 재택근무는 주 2회까지 가능합니다. 팀장 사전 승인이 필요합니다....
   → HR-003 | 출장비는 교통비, 숙박비, 식비를 포함합니다. 사전 승인 후 법인카드로 결제합니다....

Q: 출장비 정산 방법이 궁금합니다
   → HR-003 | 출장비는 교통비, 숙박비, 식비를 포함합니다. 사전 승인 후 법인카드로 결제합니다....
   → HR-006 | 건강검진은 연 1회 회사 부담으로 제공됩니다. 배우자 검진비 50%를 지원합니다....
   → HR-001 | 재택근무는 주 2회까지 가능합니다. 팀장 사전 승인이 필요합니다....

Q: 성과평가 비율이 어떻게 되나요?
   → HR-005 | 성과평가는 반기 1회 실시합니다. 상사 평가 60%, 동료 평가 20%, 자기 평가 20%...
   → HR-006 | 건강검진은 연 1회 회사 부담으로 제공됩니다. 배우자 검진비 50%를 지원합니다....
   → HR-003 | 출장비는 교통비, 숙박비, 식비를 포함합니다. 사전 승인 후 법인카드로 결제합니다....

📌 기준점(t1) 저장 완료


## 2. 문서 업데이트 시뮬레이션

3개월 후, 회사 정책이 바뀌었다:
- **HR-001** 수정: 재택근무 주 2회 → 주 3회, 수요일은 자동 승인
- **HR-003** 수정: 출장비 승인 체계 변경
- **HR-007** 신규: 유연근무제 도입
- **HR-008** 신규: 교육비 지원 제도 추가

In [4]:
# ── 문서 v2: 정책 변경 반영 ──
docs_v2 = [
    Document(
        page_content="재택근무는 주 3회까지 가능합니다. 수요일은 전사 재택근무일로 별도 승인이 필요 없습니다. 그 외 요일은 팀장 승인이 필요합니다.",
        metadata={"id": "HR-001", "title": "재택근무 정책"},
    ),
    Document(
        page_content="연차는 입사 1년 후 15일이 부여됩니다. 미사용 연차는 다음 해로 이월되지 않습니다.",
        metadata={"id": "HR-002", "title": "연차 정책"},
    ),
    Document(
        page_content="출장비는 교통비, 숙박비, 식비를 포함합니다. 50만원 이하는 팀장 승인, 초과는 부서장 승인이 필요합니다. 개인카드 사용 후 경비 청구도 가능합니다.",
        metadata={"id": "HR-003", "title": "출장비 정책"},
    ),
    Document(
        page_content="신입사원 교육은 입사 첫 주에 진행됩니다. OJT 2주, 멘토링 3개월이 포함됩니다.",
        metadata={"id": "HR-004", "title": "온보딩 프로그램"},
    ),
    Document(
        page_content="성과평가는 반기 1회 실시합니다. 상사 평가 60%, 동료 평가 20%, 자기 평가 20%입니다.",
        metadata={"id": "HR-005", "title": "성과평가 기준"},
    ),
    Document(
        page_content="건강검진은 연 1회 회사 부담으로 제공됩니다. 배우자 검진비 50%를 지원합니다.",
        metadata={"id": "HR-006", "title": "건강검진 지원"},
    ),
    Document(
        page_content="유연근무제를 시행합니다. 코어타임(10시~15시) 외 출퇴근 시간을 자유롭게 조정할 수 있습니다.",
        metadata={"id": "HR-007", "title": "유연근무제"},
    ),
    Document(
        page_content="직무 관련 외부 교육은 연 200만원까지 지원합니다. 사전 승인 후 수료 시 전액 환급됩니다.",
        metadata={"id": "HR-008", "title": "교육비 지원"},
    ),
]

vectorstore_v2 = InMemoryVectorStore.from_documents(docs_v2, embeddings)
retriever_v2 = vectorstore_v2.as_retriever(search_kwargs={"k": 3})

print(f"✅ 벡터 스토어 v2 구축 완료 — 문서 {len(docs_v2)}건 (+2건 신규)")

✅ 벡터 스토어 v2 구축 완료 — 문서 8건 (+2건 신규)


In [5]:
# ── 동일 쿼리로 재검색 (t2) ──
current = {}
for query in test_queries:
    docs = retriever_v2.invoke(query)
    current[query] = [d.metadata["id"] for d in docs]
    print(f"Q: {query}")
    for d in docs:
        print(f"   → {d.metadata['id']} | {d.page_content[:50]}...")
    print()

Q: 재택근무는 어떻게 신청하나요?
   → HR-001 | 재택근무는 주 3회까지 가능합니다. 수요일은 전사 재택근무일로 별도 승인이 필요 없습니다....
   → HR-007 | 유연근무제를 시행합니다. 코어타임(10시~15시) 외 출퇴근 시간을 자유롭게 조정할 수 있...
   → HR-006 | 건강검진은 연 1회 회사 부담으로 제공됩니다. 배우자 검진비 50%를 지원합니다....

Q: 연차 몇 일이에요?
   → HR-002 | 연차는 입사 1년 후 15일이 부여됩니다. 미사용 연차는 다음 해로 이월되지 않습니다....
   → HR-007 | 유연근무제를 시행합니다. 코어타임(10시~15시) 외 출퇴근 시간을 자유롭게 조정할 수 있...
   → HR-001 | 재택근무는 주 3회까지 가능합니다. 수요일은 전사 재택근무일로 별도 승인이 필요 없습니다....

Q: 출장비 정산 방법이 궁금합니다
   → HR-003 | 출장비는 교통비, 숙박비, 식비를 포함합니다. 50만원 이하는 팀장 승인, 초과는 부서장 ...
   → HR-006 | 건강검진은 연 1회 회사 부담으로 제공됩니다. 배우자 검진비 50%를 지원합니다....
   → HR-008 | 직무 관련 외부 교육은 연 200만원까지 지원합니다. 사전 승인 후 수료 시 전액 환급됩니...

Q: 성과평가 비율이 어떻게 되나요?
   → HR-005 | 성과평가는 반기 1회 실시합니다. 상사 평가 60%, 동료 평가 20%, 자기 평가 20%...
   → HR-006 | 건강검진은 연 1회 회사 부담으로 제공됩니다. 배우자 검진비 50%를 지원합니다....
   → HR-003 | 출장비는 교통비, 숙박비, 식비를 포함합니다. 50만원 이하는 팀장 승인, 초과는 부서장 ...



## 3. Retrieval Stability Score 계산

$$\text{Stability Score} = \frac{|\text{baseline} \cap \text{current}|}{k}$$

- **1.0**: 검색 결과가 완전히 동일 (drift 없음)
- **0.0**: 검색 결과가 완전히 바뀜 (최대 drift)

In [6]:
def retrieval_stability_score(baseline_ids: list[str], current_ids: list[str]) -> float:
    """t1과 t2의 검색 결과 겹침 비율을 계산한다."""
    overlap = set(baseline_ids) & set(current_ids)
    return len(overlap) / len(baseline_ids)


print("📊 Retrieval Stability Score")
print("=" * 60)

total_score = 0
for query in test_queries:
    score = retrieval_stability_score(baseline[query], current[query])
    total_score += score

    changed = "⚠️" if score < 1.0 else "✅"
    print(f"\n{changed} Q: {query}")
    print(f"   t1: {baseline[query]}")
    print(f"   t2: {current[query]}")
    print(f"   Stability: {score:.2f}")

avg_score = total_score / len(test_queries)
print(f"\n{'=' * 60}")
print(f"평균 Stability Score: {avg_score:.2f}")
if avg_score < 0.7:
    print("⚠️ Drift가 심각합니다. 검색 품질 재검증이 필요합니다.")
elif avg_score < 1.0:
    print("📋 일부 쿼리에서 Drift가 감지되었습니다. 영향 분석을 진행하세요.")
else:
    print("✅ 검색 결과가 안정적입니다.")

📊 Retrieval Stability Score

⚠️ Q: 재택근무는 어떻게 신청하나요?
   t1: ['HR-001', 'HR-006', 'HR-003']
   t2: ['HR-001', 'HR-007', 'HR-006']
   Stability: 0.67

⚠️ Q: 연차 몇 일이에요?
   t1: ['HR-002', 'HR-001', 'HR-003']
   t2: ['HR-002', 'HR-007', 'HR-001']
   Stability: 0.67

⚠️ Q: 출장비 정산 방법이 궁금합니다
   t1: ['HR-003', 'HR-006', 'HR-001']
   t2: ['HR-003', 'HR-006', 'HR-008']
   Stability: 0.67

✅ Q: 성과평가 비율이 어떻게 되나요?
   t1: ['HR-005', 'HR-006', 'HR-003']
   t2: ['HR-005', 'HR-006', 'HR-003']
   Stability: 1.00

평균 Stability Score: 0.75
📋 일부 쿼리에서 Drift가 감지되었습니다. 영향 분석을 진행하세요.


### 3-2. 순위 기반 평가: Rank-Biased Overlap (RBO)

단순 Overlap은 **순위를 무시**한다. 1위가 바뀐 것과 3위가 바뀐 것의 심각도가 같다.

**RBO**(Webber et al., 2010)는 상위 결과에 더 큰 가중치를 부여하는 IR 표준 지표다.
- `p` 파라미터(0~1): 1에 가까울수록 하위 순위도 중요하게 반영
- 반환값 0~1: 1이면 동일 순위, 0이면 완전히 다름

In [7]:
def rbo_score(list1: list, list2: list, p: float = 0.9) -> float:
    """Rank-Biased Overlap — 상위 결과에 가중치를 두는 순위 비교 지표."""
    k = max(len(list1), len(list2))
    l1 = list(list1) + [None] * (k - len(list1))
    l2 = list(list2) + [None] * (k - len(list2))

    score = 0.0
    for d in range(1, k + 1):
        s1 = set(l1[:d]) - {None}
        s2 = set(l2[:d]) - {None}
        if not s1 or not s2:
            continue
        score += (p ** (d - 1)) * len(s1 & s2) / d
    return (1 - p) * score


# ── Overlap vs RBO 비교 ──
print(f"{'쿼리':<35} {'Overlap':>8} {'RBO(p=.9)':>10} {'차이':>6}")
print("-" * 65)

for query in test_queries:
    b, c = baseline[query], current[query]
    overlap = retrieval_stability_score(b, c)
    rbo = rbo_score(b, c, p=0.9)
    diff = rbo - overlap
    flag = "⚠️" if abs(diff) > 0.1 else "  "
    print(f"{query:<35} {overlap:>7.2f} {rbo:>10.2f} {flag}{diff:>+.2f}")

print()
print("💡 Overlap과 RBO가 크게 다르면, 순위 변화는 있지만 구성원은 비슷한 것 (또는 그 반대)")

쿼리                                   Overlap  RBO(p=.9)     차이
-----------------------------------------------------------------
재택근무는 어떻게 신청하나요?                       0.67       0.20 ⚠️-0.47
연차 몇 일이에요?                             0.67       0.20 ⚠️-0.47
출장비 정산 방법이 궁금합니다                       0.67       0.24 ⚠️-0.42
성과평가 비율이 어떻게 되나요?                      1.00       0.27 ⚠️-0.73

💡 Overlap과 RBO가 크게 다르면, 순위 변화는 있지만 구성원은 비슷한 것 (또는 그 반대)


### 3-3. LLM Judge: 검색 품질 정성 평가

정량 지표만으로는 **"바뀐 검색 결과가 더 나은가, 나빠졌는가?"**를 판단할 수 없다.

LLM을 Judge로 활용하여 검색 결과의 **관련성(Relevance)**과 **충분성(Sufficiency)**을 직접 채점한다.

In [8]:
import re as _re


def judge_retrieval_quality(
    query: str,
    docs_a: list,
    docs_b: list,
    label_a: str = "v1",
    label_b: str = "v2",
) -> dict:
    """LLM이 두 검색 결과 세트의 품질을 비교 평가한다."""
    judge = ChatOpenAI(model="gpt-5.4", temperature=0)
    handler = CallbackHandler()

    def _fmt(docs):
        return "\n".join(f"[{i+1}] {d.page_content}" for i, d in enumerate(docs))

    prompt = f"""사용자 질문에 대해 두 검색 결과 세트를 비교 평가하세요.

[질문]
{query}

[검색 결과 A]
{_fmt(docs_a)}

[검색 결과 B]
{_fmt(docs_b)}

평가 기준 (각 1~5점):
- 관련성(Relevance): 질문에 직접 관련된 문서 비율
- 충분성(Sufficiency): 질문에 완전히 답하기 충분한 정보량
- 노이즈(Noise): 불필요한 문서가 적을수록 높은 점수

반드시 아래 형식으로만 답하세요:
A_관련성: N
A_충분성: N
A_노이즈: N
B_관련성: N
B_충분성: N
B_노이즈: N
분석: (한 문장)"""

    response = judge.invoke(
        prompt,
        config={
            "callbacks": [handler],
            "metadata": {
                "langfuse_tags": ["llm-judge", "drift-lab"],
                "langfuse_session_id": "retrieval-drift-lab",
                "evaluation_type": "retrieval_quality",
            },
        },
    )

    # 점수 파싱
    text = response.content
    scores = {}
    for line in text.strip().split("\n"):
        m = _re.match(r"([AB])_(관련성|충분성|노이즈):\s*(\d)", line)
        if m:
            scores[f"{m.group(1)}_{m.group(2)}"] = int(m.group(3))

    analysis = ""
    for line in text.strip().split("\n"):
        if line.startswith("분석:"):
            analysis = line[3:].strip()

    return {
        "query": query,
        "scores": scores,
        "analysis": analysis,
        "label_a": label_a,
        "label_b": label_b,
    }

In [9]:
# ── v1 vs v2 검색 품질 LLM 평가 ──
print("📊 LLM Judge: 검색 품질 비교 (v1 vs v2)")
print("=" * 75)

judge_results = []
for query in test_queries:
    docs_a = retriever_v1.invoke(query)
    docs_b = retriever_v2.invoke(query)
    result = judge_retrieval_quality(query, docs_a, docs_b, "v1", "v2")
    judge_results.append(result)

    s = result["scores"]
    a_avg = sum(s.get(f"A_{k}", 0) for k in ["관련성","충분성","노이즈"]) / 3
    b_avg = sum(s.get(f"B_{k}", 0) for k in ["관련성","충분성","노이즈"]) / 3
    winner = "A(v1)" if a_avg > b_avg else "B(v2)" if b_avg > a_avg else "TIE"

    print(f"\nQ: {query}")
    print(f"   v1 — 관련성:{s.get('A_관련성','?')} 충분성:{s.get('A_충분성','?')} 노이즈:{s.get('A_노이즈','?')}  (avg {a_avg:.1f})")
    print(f"   v2 — 관련성:{s.get('B_관련성','?')} 충분성:{s.get('B_충분성','?')} 노이즈:{s.get('B_노이즈','?')}  (avg {b_avg:.1f})")
    print(f"   → {winner} | {result['analysis']}")

📊 LLM Judge: 검색 품질 비교 (v1 vs v2)

Q: 재택근무는 어떻게 신청하나요?
   v1 — 관련성:2 충분성:2 노이즈:2  (avg 2.0)
   v2 — 관련성:4 충분성:4 노이즈:4  (avg 4.0)
   → B(v2) | B가 재택근무 신청 방식과 승인 조건을 더 구체적으로 담고 있어 질문에 더 직접적이고 충분하게 답하며, A는 관련 없는 문서 비중이 더 높습니다.

Q: 연차 몇 일이에요?
   v1 — 관련성:2 충분성:5 노이즈:2  (avg 3.0)
   v2 — 관련성:2 충분성:5 노이즈:2  (avg 3.0)
   → TIE | 두 세트 모두 연차 질문에 대한 핵심 답변은 포함해 충분성은 높지만, 나머지 문서가 연차와 직접 무관해 관련성과 노이즈 평가는 동일하게 낮습니다.

Q: 출장비 정산 방법이 궁금합니다
   v1 — 관련성:2 충분성:2 노이즈:2  (avg 2.0)
   v2 — 관련성:2 충분성:4 노이즈:2  (avg 2.7)
   → B(v2) | 두 결과 모두 불필요한 문서가 2개씩 포함되어 관련성과 노이즈는 비슷하지만, B는 출장비 승인 기준과 개인카드 청구 가능 여부까지 포함해 질문에 더 충분하게 답합니다.

Q: 성과평가 비율이 어떻게 되나요?
   v1 — 관련성:2 충분성:5 노이즈:2  (avg 3.0)
   v2 — 관련성:2 충분성:5 노이즈:2  (avg 3.0)
   → TIE | 두 세트 모두 성과평가 비율에 대한 정답 문서가 포함되어 충분성은 높지만, 나머지 2개 문서가 질문과 무관해 관련성과 노이즈 평가는 동일하게 낮습니다.


### 3-4. 종합 대시보드

정량(Overlap, RBO)과 정성(LLM Judge)을 한 화면에서 비교한다.
**Drift가 발생했지만 품질은 오히려 향상**된 경우와, **품질이 실제로 하락**한 경우를 구분할 수 있다.

In [10]:
# ── 종합 대시보드 ──
print(f"{'쿼리':<24} {'Overlap':>7} {'RBO':>5} {'v1 avg':>7} {'v2 avg':>7} {'판정':>8}")
print("-" * 68)

improved, degraded, stable = 0, 0, 0
for i, query in enumerate(test_queries):
    b, c = baseline[query], current[query]
    overlap = retrieval_stability_score(b, c)
    rbo = rbo_score(b, c, p=0.9)

    s = judge_results[i]["scores"]
    a_avg = sum(s.get(f"A_{k}", 0) for k in ["관련성","충분성","노이즈"]) / 3
    b_avg = sum(s.get(f"B_{k}", 0) for k in ["관련성","충분성","노이즈"]) / 3

    if b_avg > a_avg + 0.3:
        verdict = "📈 개선"
        improved += 1
    elif a_avg > b_avg + 0.3:
        verdict = "📉 하락"
        degraded += 1
    else:
        verdict = "➡️ 유지"
        stable += 1

    q_short = query[:22] + ".." if len(query) > 24 else query
    print(f"{q_short:<24} {overlap:>7.2f} {rbo:>5.2f} {a_avg:>7.1f} {b_avg:>7.1f} {verdict:>8}")

print(f"\n요약: 개선 {improved} / 유지 {stable} / 하락 {degraded}")
if degraded > 0:
    print("⚠️ 품질이 하락한 쿼리가 있습니다. 문서 변경 검토가 필요합니다.")
else:
    print("✅ 문서 업데이트가 검색 품질에 부정적 영향을 주지 않았습니다.")

쿼리                       Overlap   RBO  v1 avg  v2 avg       판정
--------------------------------------------------------------------
재택근무는 어떻게 신청하나요?            0.67  0.20     2.0     4.0     📈 개선
연차 몇 일이에요?                  0.67  0.20     3.0     3.0    ➡️ 유지
출장비 정산 방법이 궁금합니다            0.67  0.24     2.0     2.7     📈 개선
성과평가 비율이 어떻게 되나요?           1.00  0.27     3.0     3.0    ➡️ 유지

요약: 개선 2 / 유지 2 / 하락 0
✅ 문서 업데이트가 검색 품질에 부정적 영향을 주지 않았습니다.


## 4. 검색 변화가 응답에 미치는 영향

검색 결과가 달라지면 RAG 응답도 달라진다. Langfuse로 두 버전을 비교한다.

In [11]:
rag_prompt = ChatPromptTemplate.from_template(
    """다음 문서를 참고하여 질문에 답하세요. 문서에 없는 내용은 "해당 정보를 찾을 수 없습니다"라고 답하세요.

문서:
{context}

질문: {question}"""
)

llm = ChatOpenAI(model="gpt-5.4-mini", temperature=0)
langfuse = Langfuse()


def rag_answer(query: str, retriever, version_tag: str) -> tuple[str, list[str]]:
    """RAG 체인을 실행하고 응답 + 사용된 문서 ID를 반환한다."""
    handler = CallbackHandler()

    docs = retriever.invoke(query)
    doc_ids = [d.metadata["id"] for d in docs]
    context = "\n\n".join(f"[{d.metadata['id']}] {d.page_content}" for d in docs)

    chain = rag_prompt | llm
    result = chain.invoke(
        {"context": context, "question": query},
        config={
            "callbacks": [handler],
            "metadata": {
                "langfuse_tags": [f"rag-{version_tag}", "drift-lab"],
                "langfuse_session_id": "retrieval-drift-lab",
                "docs_version": version_tag,
                "retrieved_doc_ids": ",".join(doc_ids),
            },
        },
    )
    return result.content, doc_ids


# ── v1 vs v2 응답 비교 ──
print("📊 RAG 응답 비교: 문서 v1 vs v2")
print("=" * 70)

for query in test_queries:
    answer_v1, ids_v1 = rag_answer(query, retriever_v1, "v1")
    answer_v2, ids_v2 = rag_answer(query, retriever_v2, "v2")

    drift = "⚠️ DRIFT" if ids_v1 != ids_v2 else "✅ STABLE"
    print(f"\n{drift} | Q: {query}")
    print(f"   [v1 docs: {ids_v1}] → {answer_v1[:120]}")
    print(f"   [v2 docs: {ids_v2}] → {answer_v2[:120]}")

📊 RAG 응답 비교: 문서 v1 vs v2

⚠️ DRIFT | Q: 재택근무는 어떻게 신청하나요?
   [v1 docs: ['HR-001', 'HR-006', 'HR-003']] → 팀장 사전 승인이 필요합니다. 신청 절차의 구체적인 방법은 해당 정보를 찾을 수 없습니다.
   [v2 docs: ['HR-001', 'HR-007', 'HR-006']] → 재택근무는 **주 3회까지 가능**하며, **수요일은 전사 재택근무일이라 별도 승인이 필요 없습니다.**  
**그 외 요일은 팀장 승인이 필요합니다.**

⚠️ DRIFT | Q: 연차 몇 일이에요?
   [v1 docs: ['HR-002', 'HR-001', 'HR-003']] → 입사 1년 후 연차는 **15일**이 부여됩니다.
   [v2 docs: ['HR-002', 'HR-007', 'HR-001']] → 입사 1년 후 **15일**이 부여됩니다.

⚠️ DRIFT | Q: 출장비 정산 방법이 궁금합니다
   [v1 docs: ['HR-003', 'HR-006', 'HR-001']] → 출장비는 **사전 승인 후 법인카드로 결제**합니다.  
그 외의 정산 절차에 대한 내용은 **해당 정보를 찾을 수 없습니다**.
   [v2 docs: ['HR-003', 'HR-006', 'HR-008']] → 출장비는 **교통비, 숙박비, 식비**를 포함합니다.  
정산과 관련해 **개인카드 사용 후 경비 청구도 가능**합니다.  

다만, 문서에는 **구체적인 정산 절차나 제출 서류**는 나와 있지 않습니다. 해당 정보

✅ STABLE | Q: 성과평가 비율이 어떻게 되나요?
   [v1 docs: ['HR-005', 'HR-006', 'HR-003']] → 성과평가 비율은 상사 평가 60%, 동료 평가 20%, 자기 평가 20%입니다.
   [v2 docs: ['HR-005', 'HR-006', 'HR-003']] → 성과평가 비율은 상사 평가 60%, 동료 평가 20%, 자기 평가 20%입니다.


## 5. Langfuse에서 확인할 포인트

1. **Tags 필터**: `rag-v1`, `rag-v2`로 필터링하여 검색 결과와 응답 비교
2. **Metadata → `retrieved_doc_ids`**: 어떤 문서가 검색되었는지 확인
3. **Sessions → `retrieval-drift-lab`**: 전체 실험 트레이스를 한눈에 확인
4. **Latency**: 문서 수 증가(6→8건)에 따른 검색 시간 변화 관찰
5. **Token Usage**: v1 vs v2 응답의 토큰 사용량 비교 (정책이 더 상세해지면 응답도 길어지는가?)

---

## 🔧 실습: Drift 모니터링 함수 구현

**목표**: 새로운 쿼리를 추가하고, Drift가 발생하는 쿼리를 자동 식별하는 함수를 완성한다.

In [ ]:
# TODO 1: 테스트 쿼리 3개를 추가하세요
additional_queries = [
    # "???",
    # "???",
    # "???",
]

# TODO 2: 아래 함수를 완성하세요
def monitor_drift(queries: list[str], retriever_old, retriever_new, threshold: float = 0.7) -> dict:
    """
    모든 쿼리에 대해 Stability Score를 계산하고,
    threshold 미만인 쿼리를 경고 대상으로 반환한다.

    Returns:
        {"scores": {query: score}, "alerts": [query, ...], "avg_score": float}
    """
    scores = {}
    alerts = []

    for query in queries:
        old_docs = retriever_old.invoke(query)
        new_docs = retriever_new.invoke(query)
        old_ids = [d.metadata["id"] for d in old_docs]
        new_ids = [d.metadata["id"] for d in new_docs]
        score = retrieval_stability_score(old_ids, new_ids)
        scores[query] = score
        if score < threshold:
            alerts.append(query)

    avg = sum(scores.values()) / len(scores) if scores else 0
    return {"scores": scores, "alerts": alerts, "avg_score": avg}


# 실행
all_queries = test_queries + additional_queries
report = monitor_drift(all_queries, retriever_v1, retriever_v2)

print(f"평균 Stability: {report['avg_score']:.2f}")
print(f"경고 쿼리 수: {len(report['alerts'])}")
for q in report["alerts"]:
    print(f"  ⚠️ {q} (score: {report['scores'][q]:.2f})")